<a href="https://colab.research.google.com/github/avikumart/DA-DS-Questions/blob/main/PyTorch/Pytorch_model_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class CustomClassifier(nn.Module):
  def __init__(self, num_classes: int = 1000):
    super().__init__()

    self.fc1 = nn.Linear(in_features=25088, out_features=1024)
    self.fc2 = nn.Linear(in_features=1024, out_features=512)
    self.fc3 = nn.Linear(in_features=512, out_features=256)
    self.dropout = nn.Dropout(p=0.2)
    self.fc4 = nn.Linear(in_features=256, out_features=num_classes)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    x = F.relu(self.fc1(x))
    x = self.dropout(x)
    x = F.relu(self.fc2(x))
    x = self.dropout(x)
    x = F.relu(self.fc3(x))
    x = self.dropout(x)
    x = self.fc4(x)
    return x


model = CustomClassifier()
print(model)

CustomClassifier(
  (fc1): Linear(in_features=25088, out_features=1024, bias=True)
  (fc2): Linear(in_features=1024, out_features=512, bias=True)
  (fc3): Linear(in_features=512, out_features=256, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc4): Linear(in_features=256, out_features=1000, bias=True)
)


In [5]:
dataset = torch.utils.data.TensorDataset(torch.randn(1000, 25088), torch.randint(0, 10, (1000,)))

dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

In [7]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


for epoch in range(10):
  model.train()
  for inputs, targets in dataloader:
    inputs, targets = inputs.to(device), targets.to(device)
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()


# -- evaluation loop --

model.eval()
total_loss = 0.0
total_correct = 0
total_samples = 0

with torch.no_grad(): # disable gradient
  for inputs, targets in dataloader:
    inputs, targets = inputs.to(device), targets.to(device)
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    total_loss += loss.item() * inputs.size(0)
    _, predicted = torch.max(outputs.data, 1)
    total_correct += (predicted == targets).sum().item()
    total_samples += targets.size(0)